# Semptektik Topoloji

**Modül:** `pytop.symplectic_topology`  
**Kaynak:** Arnold *Mathematical Methods of Classical Mechanics*; Hofer-Zehnder *Symplectic Invariants*; McDuff-Salamon *Introduction to Symplectic Topology*

---

Bir **semptektik manifold** $(M^{2n}, \omega)$, kapalı ($d\omega = 0$) ve dejenere olmayan ($\omega^n \ne 0$) bir 2-form $\omega$ taşıyan çift boyutlu düzgün manifolddur.  
Semptektik topoloji, klasik Hamiltonian mekaniğin geometrik temelini oluşturur ve Riemannian geometriden temel bir özellikle ayrışır: **yerel semptektik değişmez yoktur** (Darboux teoremi).

**Motivasyon:**  
- Klasik mekanikte faz uzayı $(q, p)$ doğal bir semptektik yapı taşır; Liouville teoremi hacimlerin korunduğunu söyler.  
- Kähler manifoldlar (CP^n, cebirsel değişkenler) hem semptektik hem kompleks bir yapıya sahiptir.  
- Gromov'un küçülmeme teoremi, semptektomorfizmaların hacim-koruyan dönüşümlerden daha katı olduğunu kanıtlar.

Bu notebook altı bölümden oluşur:
1. **Konu** — Semptektik manifold, Lagrangian alt-manifold, Hamiltonian ve Kähler yapı tanımları
2. **Teoremler** — Darboux, Moser kararlılık, Gromov küçülmeme, Weinstein komşuluğu
3. **Fonksiyon Analizi** — `is_symplectic_manifold`, `is_lagrangian_submanifold`, `has_hamiltonian_structure`, `admits_kahler_structure`
4. **API** — `pytop.symplectic_topology` fonksiyon referansı
5. **Örnekler** — Dört senaryo: profil analizi, etiket testi, Kähler→semptektik, cephe fonksiyonları
6. **Alıştırmalar** — Kodlama ve teori

## 1. Konu

### 1.1 Semptektik Manifold

**Tanım.** $(M^{2n}, \omega)$ çifti bir **semptektik manifold**dır: $M$ düzgün $2n$-manifold, $\omega$ üzerinde
$$d\omega = 0 \quad (\text{kapalılık}), \qquad \omega^n \ne 0 \quad (\text{dejenere-olmama})$$
koşullarını sağlayan bir 2-formdur.

| Manifold | Semptektik Form | Tam mı? | Kompakt mı? |
|----------|----------------|:-------:|:-----------:|
| $\mathbb{R}^{2n}$ | $\omega_0 = \sum dx_i \wedge dp_i$ | ✓ | ✗ |
| $T^*M$ | $\omega = -d\lambda$, $\lambda = \sum p_i\,dq_i$ | ✓ | ✗ |
| $S^2$ | $r\,dA$ (alan formu) | ✗ | ✓ |
| $\mathbb{CP}^n$ | Fubini-Study $\omega_{FS}$ | ✗ | ✓ |
| $T^{2n}$ | $\omega_0$'ın inerek-indirgenmiş hali | ✗ | ✓ |

**Yerel yokluğu:** Riemannian geometride eğrilik yerel bir değişmezdir; semptektik geometride ise Darboux teoremi tüm semptektik manifoldların aynı yerel modele sahip olduğunu söyler.

### 1.2 Örnekler ve Koadjoint Yörüngeler

- **Standart $(\mathbb{R}^{2n}, \omega_0)$:** Tüm semptektik manifoldların yerel modeli. Hamiltonyen mekaniğin koordinat formu: $\dot{q}_i = \partial H/\partial p_i$, $\dot{p}_i = -\partial H/\partial q_i$.
- **Kotanjant demet $T^*M$:** Herhangi $M$ için kanonik semptektik yapı. Klasik mekaniğin faz uzayı.
- **$S^2$ (alan formu):** En küçük kompakt semptektik manifold. $SU(2)$'nin koadjoint yörüngesi.
- **KKS formu:** $G$ Lie grubu için her koadjoint yörünge $\mathcal{O}_\mu$ kanonik bir semptektik yapı taşır: $\omega^{KKS}_\mu(\mathrm{ad}^*_X\mu, \mathrm{ad}^*_Y\mu) = \mu([X,Y])$. Geometrik kuantizasyonun temelidir.

### 1.3 Lagrangian Alt-Manifoldlar

**Tanım.** $(M^{2n}, \omega)$'nın $n$-boyutlu $L$ alt-manifoldu $\omega|_L = 0$ sağlıyorsa **Lagrangian**dır. Eşdeğer olarak $TL = (TL)^{\omega\text{-perp}}$.

Örnekler: $T^*M$'in sıfır kesiti, her kotan-jant lif $T^*_x M$, $\mathbb{CP}^n \supset \mathbb{RP}^n$.

### 1.4 Hamiltonyen Vektör Alanları

**Tanım.** $(M, \omega)$ ve $H: M \to \mathbb{R}$ düzgün fonksiyon için **Hamiltonyen vektör alanı** $X_H$:
$$i_{X_H}\omega = -dH \qquad (\text{eşdeğeri: } \omega(X_H, \cdot) = -dH(\cdot)).$$
Poisson parantezi: $\{f, g\} = \omega(X_f, X_g)$. Liouville teoremi: $\omega^n$, $X_H$ akışı altında korunur.

### 1.5 Kähler Manifoldlar

**Tanım.** $(M, J, g, \omega)$ bir **Kähler manifold**dur: $J$ integrable kompleks yapı, $g$ Hermitian metrik, $\omega(X,Y) = g(JX,Y)$ kapalı. Her Kähler manifold semptektiktir; tersi genel doğru değildir.

| Özellik | Kähler | Semptektik |
|---------|:------:|:----------:|
| $d\omega = 0$ | ✓ | ✓ |
| Kompleks yapı | ✓ | ✗ (gerekmez) |
| Hard Lefschetz | ✓ | ✗ (gerekmez) |
| Hodge ayrışımı | ✓ | ✗ |
| $CP^n$, Riemann yüzeyi | ✓ | ✓ |

In [ ]:
from pytop.symplectic_topology import (
    SymplecticProfile,
    DARBOUX_THEOREM_TAGS, HAMILTONIAN_TAGS, LAGRANGIAN_TAGS,
    SYMPLECTOMORPHISM_TAGS, KAHLER_TAGS, MOSER_THEOREM_TAGS,
    GROMOV_NONSQUEEZING_TAGS, COTANGENT_BUNDLE_TAGS,
    get_named_symplectic_profiles,
    symplectic_layer_summary, symplectic_chapter_index, symplectic_type_index,
    is_symplectic_manifold, is_lagrangian_submanifold,
    has_hamiltonian_structure, admits_kahler_structure,
    classify_symplectic, symplectic_profile,
)

print("=== Semptektik Profiller ===")
for p in get_named_symplectic_profiles():
    flags = (
        f"exact={'✓' if p.is_exact else '✗'}  "
        f"Kahler={'✓' if p.is_kahler else '✗'}  "
        f"compact={'✓' if p.is_compact else '✗'}  "
        f"monotone={'✓' if p.is_monotone else '✗'}"
    )
    print(f"  {p.display_name[:60]}")
    print(f"    Tip: {p.symplectic_type:15s}  Boyut: {p.manifold_dimension}")
    print(f"    {flags}")
    print()

In [ ]:
profiles = get_named_symplectic_profiles()

compact   = [p for p in profiles if p.is_compact]
exact     = [p for p in profiles if p.is_exact]
kahler    = [p for p in profiles if p.is_kahler]
monotone  = [p for p in profiles if p.is_monotone]
lagr      = [p for p in profiles if p.has_lagrangian]

print(f"Kompakt manifoldlar   ({len(compact)}): {[p.key for p in compact]}")
print(f"Tam (exact) manifoldlar ({len(exact)}): {[p.key for p in exact]}")
print(f"Kähler manifoldlar    ({len(kahler)}): {[p.key for p in kahler]}")
print(f"Monotone manifoldlar  ({len(monotone)}): {[p.key for p in monotone]}")
print(f"Lagrangian olanlar    ({len(lagr)}): {[p.key for p in lagr]}")
print()

# Tip indeksi
print("Tip indeksi:")
for tip, keys in symplectic_type_index().items():
    print(f"  {tip:20s}: {list(keys)}")

## 2. Teoremler

### Teorem 1 — Darboux Teoremi (Darboux 1882)

> **Teorem.** Her semptektik manifold $(M^{2n}, \omega)$'nın her noktası için, o noktanın bir komşuluğunda
> $$\omega|_U = \sum_{i=1}^n dx_i \wedge dp_i$$
> koordinatları bulunur.

**Kanıt (Moser yöntemi).** İki semptektik form $\omega_0, \omega_1$ yerel olarak $[\omega_0] = [\omega_1]$ sağlar; $\omega_t = (1-t)\omega_0 + t\omega_1$ yolunu al. $d/dt(\phi_t^*\omega_t) = 0$ ODE'ini çöz: $i_{X_t}\omega_t = -\sigma_t$ (mümkün, çünkü $\omega_t$ dejenere değil). $\phi_t = $ $X_t$'nin akışı. $\square$

**Sonuç:** Riemannian geometrinin aksine, semptektik geometride yerel değişmez yoktur.

---

### Teorem 2 — Moser Kararlılık Teoremi (Moser 1965)

> **Teorem.** $M$ kompakt manifold, $\omega_t$ ($t \in [0,1]$) $[\omega_t]$ sabit olan semptektik formlar ailesi ise, $\phi_t^* \omega_t = \omega_0$ sağlayan düzgün izotopi $\phi_t$ mevcuttur.

**Sonuç:** Kompakt bir manifolddaki semptektik yapı, kohomoloji sınıfıyla belirlenir (difomorfizma üzerine kadar).

---

### Teorem 3 — Gromov Küçülmeme Teoremi (Gromov 1985)

> **Teorem.** $r > R$ ise $B^{2n}(r)$ topu $Z^{2n}(R) = B^2(R) \times \mathbb{R}^{2n-2}$ silindiri içine semptektik olarak gömülemez.

**Önemi:** Hacim-koruyan gömülemelerde bu kısıt yoktur ($n \ge 2$ için). Semptektomorfizmalar hacim-koruyan dönüşümlerden daha **katı**dır. Kanıt: $J$-holomorik eğriler (pseudoholomorphic curves).

---

### Teorem 4 — Weinstein Lagrangian Komşuluk Teoremi (Weinstein 1971)

> **Teorem.** $(M, \omega)$'da kompakt Lagrangian $L$'nin her komşuluğu, $T^*L$'nin sıfır kesiti çevresinin bir komşuluğuna semptomorfiktir.

**Sonuç:** Tüm Lagrangian alt-manifoldlar yerel olarak aynı modele sahiptir. Floer teorisi ve kuantum kohomoloji inşalarının temelidir.

In [ ]:
# Teorem özellikleri — profil verileriyle doğrulama
profiles = get_named_symplectic_profiles()

# Darboux: tüm semptektik manifoldlar yerel olarak özdeş → yerel değişmez yok
print("Darboux: tüm profillerin semptektik tipi 'sabit' bir yerel model paylaşır")
types_set = {p.symplectic_type for p in profiles}
print(f"  Farklı semptektik tip sayısı (global): {len(types_set)} → {sorted(types_set)}")
print()

# Moser: kompakt manifoldlarda semptektik yapı kohomoloji sınıfıyla belirlenir
compact_profiles = [p for p in profiles if p.is_compact]
print(f"Moser kararlılığı kapsamında kompakt profiller: {len(compact_profiles)}")
for p in compact_profiles:
    print(f"  {p.key:35s} exact={p.is_exact}  (exact=False → kohomoloji sınıfı anlamlı)")
print()

# Gromov: küçülmeme → semptektik genişlik
gromov_p = next(p for p in profiles if p.key == "gromov_nonsqueezing_example")
print(f"Gromov profili: {gromov_p.display_name}")
print(f"  exact={gromov_p.is_exact}  Kahler={gromov_p.is_kahler}  compact={gromov_p.is_compact}")
print()

# Weinstein: Lagrangian sahip profiller
lagr_profiles = [p for p in profiles if p.has_lagrangian]
print(f"Weinstein Lagrangian komşuluk teoremi kapsamı: {len(lagr_profiles)} profil")
for p in lagr_profiles:
    print(f"  {p.key}")

## 3. Fonksiyon Analizi

### `is_symplectic_manifold(space)`

**Pseudo-kod:**
1. `tags = _extract_tags(space)` — uzaydan etiket kümesini al
2. Açık semptektik etiketi (`symplectic_manifold`, `symplectic_form`) → `True` (theorem)
3. Darboux etiketleri (`darboux_chart`, `darboux_theorem`) → `True` (yerel standart form)
4. Hamiltonyen etiketleri (`hamiltonian_vector_field`, `poisson_bracket`, ...) → `True` (semptektik gerektirir)
5. Türetilmiş yapı etiketleri (kotanjant demet, Kähler, koadjoint yörünge) → `True`
6. Engel etiketleri (`odd_dimensional`, `degenerate_form`) → `False`
7. Yoksa → `Unknown` (symbolic)

**Karmaşıklık:**
- Zaman: $O(|\text{tags}|)$ — küme kesişim işlemi
- Alan: $O(1)$ — sabit boyutlu etiket kümeleri

**Avantajlar:**
- Hızlı; beş aşamalı karar hiyerarşisi sayesinde birden fazla örtülü kaynaktan sonuca ulaşır.
- Kotanjant demet, Kähler ve koadjoint yörüngeler gibi kanonik yapıları otomatik tanır.

**Dezavantajlar / Sınırlamalar:**
- Etiket tabanlı: koordinat ya da metrikten semptektik yapıyı doğrudan saptayamaz.
- Çelişkili etiketler varsa ilk eşleşen karar geçerlidir (öncelik sırası sabit).

**Nasıl aşılır / Geliştirme fırsatları:**
- `metadata["omega"]` alanında sembolik 2-form verisi saklanarak `d omega = 0` ve `omega^n != 0` doğrudan kontrol edilebilir.

In [ ]:
class _Space:
    def __init__(self, *tags, rep="test"):
        self.tags = set(tags)
        self.representation = rep

print("is_symplectic_manifold — karar katmanları:")
test_cases = [
    ("symplectic_manifold",),
    ("darboux_chart",),
    ("hamiltonian_vector_field",),
    ("cotangent_bundle",),
    ("kahler_manifold",),
    ("coadjoint_orbit",),
    ("odd_dimensional",),
    ("degenerate_form",),
    ("unrelated_tag",),
]

print(f"  {'Etiket(ler)':<30}  {'Sonuç':<10}  {'Kriter'}")
print("  " + "─" * 70)
for tags in test_cases:
    r = is_symplectic_manifold(_Space(*tags))
    crit = r.metadata.get("criterion", "—")
    print(f"  {str(tags):<30}  {str(r.value):<10}  {crit}")

### `is_lagrangian_submanifold(space)`

**Pseudo-kod:**
1. Açık Lagrangian etiketi (`lagrangian_submanifold`, `maslov_index`, `floer_theory`, ...) → `True`
2. Bilinen Lagrangian yapılar (`zero_section_lagrangian`, `lagrangian_torus`, `lagrangian_fibration`) → `True`
3. Semptektik alt-manifold etiketleri (`symplectic_submanifold`) → `False` (Lagrangian'la çelişkili)
4. Yoksa → `Unknown`

**Karmaşıklık:** $O(|\text{tags}|)$ zaman, $O(1)$ alan.

**Avantajlar:**
- Weinstein neighborhod, Maslov indeksi ve Floer teorisi bağlantılarını tanır.
- Semptektik ↔ Lagrangian çelişkisini doğru şekilde reddeder.

**Dezavantajlar / Sınırlamalar:**
- $\omega|_L = 0$ ve $\dim L = n$ koşullarını koordinat verisiyle doğrulamaz.

**Nasıl aşılır:** `metadata["dim"]` ve `metadata["restriction"]` alanlarından boyut ve form kısıtlaması kontrol edilebilir.

In [ ]:
print("is_lagrangian_submanifold — karar katmanları:")
test_cases = [
    ("lagrangian_submanifold",),
    ("maslov_index",),
    ("floer_theory",),
    ("zero_section_lagrangian",),
    ("cotangent_fiber_lagrangian",),
    ("lagrangian_torus",),
    ("symplectic_submanifold",),
    ("non_lagrangian",),
    ("unrelated_tag",),
]

print(f"  {'Etiket(ler)':<32}  {'Sonuç':<20}  {'Kriter'}")
print("  " + "─" * 75)
for tags in test_cases:
    r = is_lagrangian_submanifold(_Space(*tags))
    crit = r.metadata.get("criterion", "—")
    print(f"  {str(tags):<32}  {str(r.value):<20}  {crit}")

### `has_hamiltonian_structure(space)`

**Pseudo-kod:**
1. Açık Hamiltonyen etiketi (`hamiltonian_vector_field`, `poisson_bracket`, `liouville_theorem`, ...) → `True`
2. Semptektik yapı etiketleri (`symplectic_manifold`, `cotangent_bundle`, `kahler_manifold`, ...) → `True` (her semptektik manifold Hamiltonyen kabul eder)
3. Semptektik yapı engelleyen etiket (`odd_dimensional`, `degenerate_form`) → `False`
4. Yoksa → `Unknown`

**Karmaşıklık:** $O(|\text{tags}|)$ zaman, $O(1)$ alan.

**Avantajlar:**
- Semptektik → Hamiltonyen imlikasyonunu doğru yansıtır.
- Noether teoremi ve korunum yasalarını tanır.

**Dezavantajlar / Sınırlamalar:**
- Belirli bir $H$ fonksiyonunun var olduğunu değil, Hamiltonyen yapısına elverişli bir manifold olduğunu kontrol eder.

In [ ]:
print("has_hamiltonian_structure — karar katmanları:")
test_cases = [
    ("hamiltonian_vector_field",),
    ("poisson_bracket",),
    ("liouville_theorem",),
    ("noether_theorem",),
    ("symplectic_manifold",),
    ("cotangent_bundle",),
    ("coadjoint_orbit",),
    ("odd_dimensional",),
    ("no_symplectic_structure",),
    ("unrelated_tag",),
]

print(f"  {'Etiket(ler)':<30}  {'Sonuç':<22}  {'Kriter'}")
print("  " + "─" * 80)
for tags in test_cases:
    r = has_hamiltonian_structure(_Space(*tags))
    crit = r.metadata.get("criterion", "—")
    print(f"  {str(tags):<30}  {str(r.value):<22}  {crit}")

### `admits_kahler_structure(space)`

**Pseudo-kod:**
1. Açık Kähler etiketi (`kahler_manifold`, `kahler_form`, `fubini_study`, `hard_lefschetz`, ...) → `True`
2. Bilinen Kähler manifoldlar (`complex_projective`, `riemann_surface`, `algebraic_variety`, `coadjoint_orbit`) → `True`
3. Açık Kähler olmayan etiket (`non_kahler`, `kodaira_thurston`, `no_hard_lefschetz`) → `False`
4. Yoksa → `Unknown`

**Karmaşıklık:** $O(|\text{tags}|)$ zaman, $O(1)$ alan.

**Avantajlar:**
- Kähler için klasik sonuçları (cebirsel değişkenler, Riemann yüzeyleri, Borel-Weil) tanır.
- Semptektik-ama-Kähler-değil durumunu (Kodaira-Thurston manifoldu) doğru reddeder.

**Dezavantajlar / Sınırlamalar:**
- Uyumlu bir kompleks yapının varlığını koordinatlardan doğrulamaz.
- Tüm Kähler manifoldlar semptektiktir ama bu fonksiyon Kähler'i ayrıca kontrol eder; iki fonksiyonu bağımsız çalıştırmak gerekir.

**Nasıl aşılır:** Hodge sayıları $h^{p,q}$ ve Hard Lefschetz ihlalini `metadata` üzerinden test eden bir katman eklenebilir.

In [ ]:
print("admits_kahler_structure — karar katmanları:")
test_cases = [
    ("kahler_manifold",),
    ("fubini_study",),
    ("hard_lefschetz",),
    ("complex_projective",),
    ("riemann_surface",),
    ("algebraic_variety",),
    ("coadjoint_orbit",),
    ("non_kahler",),
    ("kodaira_thurston",),
    ("symplectic_non_kahler",),
    ("unrelated_tag",),
]

print(f"  {'Etiket(ler)':<27}  {'Sonuç':<17}  {'Kriter'}")
print("  " + "─" * 72)
for tags in test_cases:
    r = admits_kahler_structure(_Space(*tags))
    crit = r.metadata.get("criterion", "—")
    print(f"  {str(tags):<27}  {str(r.value):<17}  {crit}")

## 4. API Referansı

| Fonksiyon / Sınıf | Açıklama | Döndürür |
|-------------------|----------|----------|
| `SymplecticProfile` | Semptektik manifold öğretim profili | `dataclass(frozen=True)` |
| `get_named_symplectic_profiles()` | 8 kanonik profili döndürür | `tuple[SymplecticProfile, ...]` |
| `symplectic_layer_summary()` | Katman → profil sayısı | `dict[str, int]` |
| `symplectic_chapter_index()` | Bölüm → anahtar listesi | `dict[str, tuple[str, ...]]` |
| `symplectic_type_index()` | Tip → anahtar listesi | `dict[str, tuple[str, ...]]` |
| `DARBOUX_THEOREM_TAGS` | Darboux teoremi etiketleri | `frozenset[str]` |
| `HAMILTONIAN_TAGS` | Hamiltonyen yapı etiketleri | `frozenset[str]` |
| `LAGRANGIAN_TAGS` | Lagrangian alt-manifold etiketleri | `frozenset[str]` |
| `SYMPLECTOMORPHISM_TAGS` | Semptektomorfizm etiketleri | `frozenset[str]` |
| `KAHLER_TAGS` | Kähler manifold etiketleri | `frozenset[str]` |
| `MOSER_THEOREM_TAGS` | Moser kararlılık etiketleri | `frozenset[str]` |
| `GROMOV_NONSQUEEZING_TAGS` | Gromov küçülmeme etiketleri | `frozenset[str]` |
| `COTANGENT_BUNDLE_TAGS` | Kotanjant demet etiketleri | `frozenset[str]` |
| `is_symplectic_manifold(space)` | Semptektik yapı kontrolü | `Result` |
| `is_lagrangian_submanifold(space)` | Lagrangian alt-manifold kontrolü | `Result` |
| `has_hamiltonian_structure(space)` | Hamiltonyen yapı kontrolü | `Result` |
| `admits_kahler_structure(space)` | Kähler yapı kontrolü | `Result` |
| `classify_symplectic(space)` | Tüm analizleri tek sözlükte toplar | `dict[str, Result]` |
| `symplectic_profile(space)` | Kapsamlı semptektik profil | `dict[str, Any]` |

In [ ]:
# Özet ve indeks fonksiyonları
print("symplectic_layer_summary():")
for k, v in symplectic_layer_summary().items():
    print(f"  {k}: {v}")

print()
print("symplectic_chapter_index():")
for ch, keys in symplectic_chapter_index().items():
    print(f"  Bölüm {ch}: {len(keys)} profil")

print()
print("Etiket küme boyutları:")
tag_sets = [
    ("DARBOUX",         DARBOUX_THEOREM_TAGS),
    ("HAMILTONIAN",     HAMILTONIAN_TAGS),
    ("LAGRANGIAN",      LAGRANGIAN_TAGS),
    ("SYMPLECTO.",      SYMPLECTOMORPHISM_TAGS),
    ("KAHLER",          KAHLER_TAGS),
    ("MOSER",           MOSER_THEOREM_TAGS),
    ("GROMOV",          GROMOV_NONSQUEEZING_TAGS),
    ("COTANGENT",       COTANGENT_BUNDLE_TAGS),
]
for name, ts in tag_sets:
    print(f"  {name:<15}: {len(ts)} etiket")

## 5. Örnekler

In [ ]:
# Örnek 1: Profillerin semptektik özelliklerini tam tablo olarak göster
profiles = get_named_symplectic_profiles()

print(f"{'Profil anahtarı':<35} {'Tip':<16} {'Exact'} {'Kähler'} {'Kompakt'} {'Monotone'}")
print("─" * 95)
for p in profiles:
    e = "✓" if p.is_exact   else "✗"
    k = "✓" if p.is_kahler  else "✗"
    c = "✓" if p.is_compact else "✗"
    m = "✓" if p.is_monotone else "✗"
    print(f"  {p.key:<33} {p.symplectic_type:<16} {e:^5}  {k:^6}  {c:^7}  {m:^8}")
print("─" * 95)

# İlişkiler
print()
n = len(profiles)
print("İlişki kontrolleri:")
kahler_and_not_monotone = [p for p in profiles if p.is_kahler and not p.is_monotone]
print(f"  Kähler ama monotone değil: {[p.key for p in kahler_and_not_monotone]}")
exact_and_compact = [p for p in profiles if p.is_exact and p.is_compact]
print(f"  Tam (exact) ve kompakt   : {[p.key for p in exact_and_compact]}  (beklenen: boş)")

In [ ]:
# Örnek 2: Etiket tabanlı semptektik test — gerçek uygulamada nasıl kullanılır
test_spaces = [
    _Space("cotangent_bundle", "lagrangian_torus",      rep="T*M with Lagrangian fiber"),
    _Space("kahler_manifold",  "complex_projective",    rep="CP^n"),
    _Space("coadjoint_orbit",  "riemann_surface",       rep="S^2 as orbit of SU(2)"),
    _Space("gromov_nonsqueezing", "symplectic_width",   rep="Gromov width domain"),
    _Space("moser_theorem",    "symplectomorphism",     rep="Moser isotopy"),
    _Space("odd_dimensional",                           rep="S^3"),
    _Space("non_kahler",       "symplectic_manifold",   rep="Kodaira-Thurston"),
]

print(f"{'Uzay':<35} {'Semptektik':<10} {'Lagrangian':<12} {'Hamiltonyen':<12} {'Kähler'}")
print("─" * 90)
for s in test_spaces:
    sym  = is_symplectic_manifold(s).value
    lag  = is_lagrangian_submanifold(s).value
    ham  = has_hamiltonian_structure(s).value
    kah  = admits_kahler_structure(s).value
    # Kısa etiketler
    def fmt(v):
        if v == "symplectic_manifold": return "✓"
        if v == "lagrangian_submanifold": return "✓"
        if v == "hamiltonian_structure": return "✓"
        if v == "kahler_structure": return "✓"
        if v == "unknown": return "?"
        return "✗"
    print(f"  {s.representation:<33} {fmt(sym):^10} {fmt(lag):^12} {fmt(ham):^12} {fmt(kah)}")

In [ ]:
# Örnek 3: Kähler → Semptektik imlikasyonu ve Gromov küçülmemesi bağlantısı
print("=== Kähler → Semptektik İmlikasyonu ===")
kahler_spaces = [
    _Space("kahler_manifold", rep="Genel Kähler"),
    _Space("fubini_study",    rep="CP^n (Fubini-Study)"),
    _Space("riemann_surface", rep="Riemann yüzeyi"),
    _Space("algebraic_variety", rep="Cebirsel değişken"),
]

for s in kahler_spaces:
    sym = is_symplectic_manifold(s)
    kah = admits_kahler_structure(s)
    print(f"  {s.representation:<25}  Kähler={kah.is_true}  Semptektik={sym.is_true}")

print()
print("=== Gromov Küçülmemesi: Semptektik Genişlik ===")
gromov_p = next(p for p in get_named_symplectic_profiles() if p.key == "gromov_nonsqueezing_example")
print(f"  Profil: {gromov_p.display_name}")
print(f"  Boyut : {gromov_p.manifold_dimension}")
print(f"  Exact : {gromov_p.is_exact}")
print("  Teorem: B^{2n}(r) ⊄_semptektik B^2(R) x R^{2n-2} for r > R")
# Etiket tabanlı kontrol
gromov_s = _Space("gromov_nonsqueezing", "symplectic_capacities")
r = is_symplectic_manifold(gromov_s)
print(f"  is_symplectic_manifold({gromov_s.tags}): {r.value}")

In [ ]:
# Örnek 4: classify_symplectic ve symplectic_profile cephe fonksiyonları
print("=== classify_symplectic ===")
rich_space = _Space(
    "kahler_manifold", "complex_projective", "fubini_study",
    "lagrangian_submanifold", "hamiltonian_flow",
    rep="CP^n (tam analiz)"
)

cls = classify_symplectic(rich_space)
for key, result in cls.items():
    status = "✓" if result.is_true else ("✗" if result.is_false else "?")
    crit = result.metadata.get("criterion", "—")
    print(f"  {key:<32} {status}  ({crit})")

print()
print("=== symplectic_profile ===")
prof = symplectic_profile(rich_space)
print(f"  Uzay            : {prof['space'].representation}")
print(f"  Etiketler       : {prof['tags']}")
print(f"  Representation  : {prof['representation']}")
print(f"  Özet:")
for k, v in prof["summary"].items():
    print(f"    {k:<32}: {v}")

## 6. Alıştırmalar

**Alıştırma 1.**  
`get_named_symplectic_profiles()` profillerini kullanarak:  
(a) `is_exact=True` olan profillerin `is_compact=False` olduğunu kontrol edin ve nedenini açıklayın: kompakt kapanıklı manifoldda $[\omega]^n \ne 0$ iken tam bir $\omega = d\alpha$ neden mümkün olmaz?  
(b) `is_monotone=True` olan profilleri listeleyin; Floer teorisinin monotone koşula neden ihtiyaç duyduğunu `cpn_fubini_study` profilinin `focus` alanından okuyarak açıklayın.  
(c) `coadjoint_orbit_su2` profilinin KKS formu ile $S^2$ alan formu arasındaki ilişkiyi `focus` alanından bulun: $r = 1$ için $[\omega^{KKS}] = ?$ eşitliği ne anlama gelir?

**Alıştırma 2.**  
`is_symplectic_manifold` ve `admits_kahler_structure` fonksiyonlarını birlikte kullanarak:  
(a) `_Space("kahler_manifold")` ve `_Space("non_kahler", "symplectic_manifold")` uzayları için her iki fonksiyonu çalıştırın; "Kähler ama semptektik-değil" ve "semptektik ama Kähler-değil" durumlarının olup olmadığını kontrol edin.  
(b) `KAHLER_TAGS` ve `DARBOUX_THEOREM_TAGS` kesişimini hesaplayın ve Darboux teoreminin Kähler manifoldlara özgü bir şey söylemediğini (kesişimin boş olmasını beklediğinizi) doğrulayın.  
(c) Hard Lefschetz ihlali olan bir manifoldu `_Space("no_hard_lefschetz")` ile test edin; `admits_kahler_structure` doğru reddettiğini gösterin.

**Alıştırma 3.**  
`is_lagrangian_submanifold` ve `has_hamiltonian_structure` fonksiyonlarını analiz ederek:  
(a) Kotanjant demet $T^*M$'nin hem Lagrangian (`zero_section_lagrangian`) hem de Hamiltonyen (`canonical_symplectic`) yapıya sahip olduğunu etiket tabanlı olarak doğrulayın.  
(b) Semptektik bir alt-manifold (`symplectic_submanifold`) ile Lagrangian bir alt-manifoldin neden birbirini dışladığını matematiksel olarak açıklayın; `is_lagrangian_submanifold(_Space("symplectic_submanifold"))` çıktısını inceleyin.  
(c) `_Space("lagrangian_fibration", "hamiltonian_flow")` örneğiyle Arnold-Liouville teoremini (integre Hamiltonyen sistemlerde faz uzayı Lagrangian torus fibration'a sahiptir) `has_hamiltonian_structure` ve `is_lagrangian_submanifold` çıktılarıyla ilişkilendirin.

**Alıştırma 4.**  
`classify_symplectic` ve `symplectic_profile` cephe fonksiyonları için:  
(a) `_Space("gromov_nonsqueezing", "symplectic_capacities")` uzayı için `classify_symplectic` çalıştırın; hangi analizlerin `True`, hangilerinin `Unknown` döndürdüğünü ve bunların Gromov teoreminin içeriğiyle uyumunu açıklayın.  
(b) `moser_stability_example` profilinin `focus` alanındaki kanıt adımlarını okuyun ve `_Space("moser_theorem", "cohomologous_forms")` üzerinde `classify_symplectic` çalıştırarak Moser izotopisinin hangi yapılara karşılık geldiğini gösterin.  
(c) `symplectic_type_index()` çıktısını kullanarak her tipin (`standard`, `cotangent`, `kahler`, `coadjoint_orbit`, `product`) kaç profil içerdiğini listeleyin ve her tipin Darboux teoremiyle ilişkisini ("hepsi yerel olarak özdeşse, global ayrım nerede?" sorusuyla) tartışın.

In [ ]:
# Alıştırma 4c kurulumu — tip indeksi + roller
type_index = symplectic_type_index()

darboux_notu = {
    "standard"       : "Darboux'nun standart modeli — tüm yerel modeller buna eşdeğer",
    "cotangent"      : "Kanonik tam yapı; yerel model = standart R^{2n}",
    "kahler"         : "Ekstra kompleks yapı; yerel Darboux modeli paylaşılır ama global fark var",
    "coadjoint_orbit": "KKS formu; yörünge boyutuna göre Darboux koordinatları değişir",
    "product"        : "Çarpım semptektik form; her faktör kendi Darboux koordinatlarını taşır",
}

print(f"{'Tip':<18}  {'Profil':>6}  {'Anahtarlar':<35}  Darboux ilişkisi")
print("─" * 110)
total = 0
for tip, keys in type_index.items():
    not_ = darboux_notu.get(tip, "—")
    print(f"  {tip:<16}  {len(keys):>6}  {str(list(keys)):<35}  {not_}")
    total += len(keys)
print("─" * 110)
print(f"  {'Toplam':<16}  {total:>6}")

---

## Özet

| Kavram | Ana Sonuç |
|--------|-----------|
| Semptektik manifold | $(M^{2n}, \omega)$: kapalı ($d\omega=0$) ve dejenere olmayan ($\omega^n\ne 0$) 2-form |
| Darboux teoremi | Yerel semptektik değişmez yoktur; tüm modeller $\omega_0 = \sum dx_i\wedge dp_i$'ye eşdeğer |
| Moser kararlılığı | Kompakt manifoldda aynı kohomoloji sınıfındaki semptektik formlar difomorfik |
| Lagrangian | $L^n \subset M^{2n}$: $\omega|_L = 0$; Weinstein komşuluğu $T^*L$'ye eşdeğer |
| Hamiltonyen yapı | $i_{X_H}\omega = -dH$; her semptektik manifold Hamiltonyen kabul eder |
| Kähler manifold | Semptektik + kompleks + Hermitian uyumluluk; Hard Lefschetz geçerli |
| Gromov küçülmeme | $B^{2n}(r)\not\hookrightarrow Z^{2n}(R)$ semptektik ($r>R$); semptektomorfizmalar hacimden katıdır |
| Koadjoint yörüngesi | KKS formuyla kanonik semptektik yapı; geometrik kuantizasyonun temeli |

**Bir sonraki adım:** `pytop.motivic_homotopy` — motivik homotopi teorisi, $\mathbb{A}^1$-homotopi, Voevodsky.